# Structuted Output in LLMs

- LLM can generate any output, but, in plaintext or markdown format.
- But, we sometime need data in different structured data formats like JSON, XML, Markdown (default), HTML, etc. from the LLMs.
- Python provides some libraries to allow us to define a schema and LLM will forcibly receive data in specified output format.
- We'll only focus on JSON for now.

In [1]:
from IPython.display import Markdown # Just to print AI response Beautifully

from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

user_input = "Tea"

## Approach 1: Specify rules in Prompt

- We can just specify the rules to return a specific format in the prompt itself and the LLM will listen.
- But this is the most unreliable way to do so.

Reasons (see the output below first):
- We just instructed to return list of `ingredients` and `steps`, but the LLM also returned `description`, `prepTime`, `cookTime`, `servings`, `notes`, etc.
- For `ingredients`, we just asked a list of strings, but, the LLM returned a list of objects having keys like `item`, `quantity`, and `unit`.
- We just asked the LLM to return a **list** of steps, but, it returned a list of objects having keys like `stepNumber` and `instructions`
- Thus, the LLM may assume extra information, and may return the output in an unexpected way.
- Plus, the LLM may also return extra information like "Here's the data...", "May, I change the format..."
- Plus, there are some LLMs that are not capable of directly generating structured outputs, and rely on other Python libraries.

In [ ]:
result = llm.invoke(f"""
    You are an expert cook and recipe generator.
    Give me a recipe to cook/prepare: {user_input}
    Give me information related to:
    - List of Ingredients
    - List of Steps in a sequence
    IMPORTANT: The output should be in a JSON format
""")

# Try uncommenting and running again, the output will be again different this time.
# result

```json
{
  "recipeName": "Classic Hot Tea",
  "description": "A comforting and timeless beverage, perfect for any time of day. This recipe focuses on a basic preparation, allowing for personal customization.",
  "prepTime": "2 minutes",
  "cookTime": "3-5 minutes",
  "servings": "1",
  "ingredients": [
    {
      "item": "Water",
      "quantity": "1 cup",
      "unit": ""
    },
    {
      "item": "Tea Bag (Black, Green, Herbal, etc.) OR Loose Leaf Tea",
      "quantity": "1",
      "unit": "bag"
    },
    {
      "item": "Milk (Optional)",
      "quantity": "To taste",
      "unit": ""
    },
    {
      "item": "Sugar, Honey, or Sweetener (Optional)",
      "quantity": "To taste",
      "unit": ""
    }
  ],
  "steps": [
    {
      "stepNumber": 1,
      "instruction": "Boil the Water: Heat 1 cup of fresh cold water in a kettle or saucepan until it reaches a rolling boil. For optimal taste, avoid re-boiling water."
    },
    {
      "stepNumber": 2,
      "instruction": "Prepare the Mug/Teapot: While the water is heating, place your chosen tea bag into a mug, or add 1 teaspoon of loose leaf tea to a tea infuser and place it in your mug/teapot."
    },
    {
      "stepNumber": 3,
      "instruction": "Pour and Steep: Once the water has boiled, carefully pour the hot water over the tea bag or loose leaf tea in your mug/teapot. Ensure the tea is fully submerged."
    },
    {
      "stepNumber": 4,
      "instruction": "Infuse the Tea: Allow the tea to steep for 3-5 minutes. Adjust steeping time based on your preferred strength and the type of tea (e.g., black tea generally steeps longer than green tea). Do not over-steep, as it can make the tea bitter."
    },
    {
      "stepNumber": 5,
      "instruction": "Remove Tea: Carefully remove the tea bag from the mug, or lift out the tea infuser with the loose leaves. Gently squeeze the tea bag against the side of the mug to extract maximum flavor, if desired."
    },
    {
      "stepNumber": 6,
      "instruction": "Customize (Optional): If you prefer, add milk (dairy or non-dairy) and/or a sweetener like sugar or honey to taste. Stir well until dissolved."
    },
    {
      "stepNumber": 7,
      "instruction": "Serve: Your delicious hot tea is ready! Serve immediately and enjoy."
    }
  ],
  "notes": "The ideal water temperature varies for different tea types: 200-212°F (93-100°C) for black tea, 175-185°F (79-85°C) for green tea, and 190-200°F (88-93°C) for oolong tea. Herbal teas can generally handle boiling water. Experiment with different tea types and steeping times to find your perfect cup."
}
```

## Approach 2: Python's Pydantic Library

> NOTE: Pydantic is NOT a LangChain Library, it comes from core Python libraries

- In Pydantic, we define a schema, **datatypes**, rules, descriptions, etc. and the LLM will strictly follows the same structure.
- In case the LLM fails to generate same output and gives something inaccurate, the app will throw an error. (This is a disadvantage, because, we don't want our App to directly crash.)

In [2]:
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    name: str = Field(description="The name of the Recipe as it is as input")
    ingredients: list[str]
    steps: list[str] = Field(description="Steps for how to cook the specified recipe in a sequence")
    notes: list[str] | None = Field(
        default=None,
        description="Notes, Warnings, or Tips as extra information",
        max_length=3
    )

"""
----- Field() in Pydantic & its important parameters ----

Field() is used to add metadata, validation rules, and descriptions to schema fields.

Key Parameters:
1. description: Explains what the field represents to the LLM (guides LLM output generation).
2. default / default_factory: Sets a default value if the field is optional (e.g., default=None).
3. max_length / min_length: Limits the maximum/minimum number of items in a list or characters in a string (e.g., max_length=3).
4. gt, ge, lt, le: Numerical constraints for numbers (e.g., ge=1 for rating >= 1).
"""

llm_with_schema = llm.with_structured_output(Recipe)
pydantic_object = llm_with_schema.invoke(f"Generate a recipe for {user_input}")
pydantic_object

Recipe(name='Tea', ingredients=['Water', 'Tea bag or loose leaf tea', 'Sugar (optional)', 'Milk (optional)'], steps=['Boil water to the desired temperature (e.g., 200-212°F or 93-100°C for black tea, cooler for green tea).', 'Place a tea bag or tea infuser with loose leaf tea into a mug.', 'Pour the hot water over the tea bag or leaves.', 'Steep for 2-5 minutes, depending on the type of tea and desired strength. Remove the tea bag or infuser.', 'Add sugar and/or milk to taste, if desired.', 'Stir and enjoy.'], notes=None)

The Output returned is as expected, but it is as a Pydantic Object.

To convert it to Python Dictionary (JSON), we use `model_dump()` method:

In [3]:
pydantic_object.model_dump()

{'name': 'Tea',
 'ingredients': ['Water',
  'Tea bag or loose leaf tea',
  'Sugar (optional)',
  'Milk (optional)'],
 'steps': ['Boil water to the desired temperature (e.g., 200-212°F or 93-100°C for black tea, cooler for green tea).',
  'Place a tea bag or tea infuser with loose leaf tea into a mug.',
  'Pour the hot water over the tea bag or leaves.',
  'Steep for 2-5 minutes, depending on the type of tea and desired strength. Remove the tea bag or infuser.',
  'Add sugar and/or milk to taste, if desired.',
  'Stir and enjoy.'],
 'notes': None}

## Approach 3: Python's TypedDict

> NOTE: TypedDict is NOT a LangChain Library, it comes from core Python libraries (`typing` module)

- In TypedDict, we define a schema and **datatypes** for dictionary keys, and the LLM will strictly follow the same structure.
- Unlike Pydantic, TypedDict doesn't throw any error if something goes wrong. The Developer may handle wrong outputs on their own.

In [4]:
from typing import TypedDict, Annotated, Literal

class Recipe(TypedDict):
    name: Annotated[str, "The name of the Recipe as it is as input"]
    ingredients: list[str]
    steps: Annotated[list[str], "Steps for how to cook the specified recipe in a sequence"]
    notes: Annotated[list[str] | None, "Notes, Warnings, or Tips as extra information"]
    difficulty: Literal["Easy", "Medium", "Hard"]

llm_with_schema = llm.with_structured_output(Recipe)
typeddict_object = llm_with_schema.invoke(f"Generate a recipe for {user_input}")
typeddict_object

{'name': 'Tea',
 'ingredients': ['Tea bag or loose leaf tea',
  'Hot water',
  'Sugar (optional)',
  'Milk (optional)'],
 'steps': ['Boil water.',
  'Place tea bag or loose leaf tea in a cup.',
  'Pour hot water over the tea.',
  'Steep for 3-5 minutes, or to desired strength.',
  'Remove tea bag or strain loose leaves.',
  'Add sugar and milk if desired, and stir.'],
 'notes': ['Adjust steeping time for stronger or weaker tea.',
  'Different types of tea (black, green, herbal) may have different ideal brewing temperatures and steeping times.'],
 'difficulty': 'Easy'}

The Output returned is directly a standard Python Dictionary (`dict`). Since it is already a Python Dictionary, no conversion method (like `model_dump()`) is needed:

# Nesting Pydantic Structures

In [6]:
from typing import Literal
from pydantic import BaseModel, Field

class Ingredients(BaseModel):
    name: str
    quantity: float
    unit: str

class Steps(BaseModel):
    number: int
    instruction: str
    importance: Literal["Critical", "May Skip"]

class Recipe(BaseModel):
    name: str
    ingredients: list[Ingredients]
    steps: list[str]

llm_with_schema = llm.with_structured_output(Recipe)
pydantic_object = llm_with_schema.invoke(f"Generate a recipe for {user_input}")
pydantic_object.model_dump()

{'name': 'Tea',
 'ingredients': [{'name': 'Water', 'quantity': 1.0, 'unit': 'cup'},
  {'name': 'Black tea bag', 'quantity': 1.0, 'unit': 'unit'}],
 'steps': ['Boil water.',
  'Place tea bag in a mug.',
  'Pour hot water over the tea bag.',
  'Steep for 3-5 minutes.',
  'Remove tea bag and discard.',
  'Serve hot.']}